In [1]:
%pwd

'd:\\Siam\\Chicken-Disease-Classification\\research'

In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Siam\\Chicken-Disease-Classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list



@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [6]:
from chicken_disease_classification.constants import *
from chicken_disease_classification.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config["artifacts_root"]])

    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
        config = self.config["prepare_callbacks"]
        model_ckpt_dir = os.path.dirname(config["checkpoint_model_filepath"])
        create_directories([
            Path(model_ckpt_dir),
            Path(config["tensorboard_root_log_dir"])
        ])

        prepare_callbacks_config = PrepareCallbacksConfig(
            root_dir=Path(config["root_dir"]),
            tensorboard_root_log_dir=Path(config["tensorboard_root_log_dir"]),
            checkpoint_model_filepath=Path(config["checkpoint_model_filepath"])
        )
        return prepare_callbacks_config
    
    def get_training_config(self) -> TrainingConfig:
        config = self.config["training"]
        self.params = self.params
        prepare_base_model = self.config["prepare_base_model"]

        training_data = os.path.join(
            self.config["data_ingestion"]["unzip_dir"],
            "Chicken-fecal-images" 
        )

        create_directories([Path(config["root_dir"])])

        training_config = TrainingConfig(
            root_dir=Path(config["root_dir"]),
            trained_model_path=Path(config["trained_model_path"]),
            updated_base_model_path=Path(prepare_base_model["updated_base_model_path"]),
            training_data=Path(training_data),
            params_epochs=self.params["EPOCHS"],
            params_batch_size=self.params["BATCH_SIZE"],
            params_is_augmentation=self.params["AUGMENTATION"],
            params_image_size=self.params["IMAGE_SIZE"]
        )
        return training_config

In [8]:
import time
import os
import sys
import tensorflow as tf
from chicken_disease_classification.logger import logging
from chicken_disease_classification.exception import CustomException

In [9]:
class PrepareCallbacksTrainingPipeline:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config

    
    @property
    def _create_tb_callback(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_run_log_dir = os.path.join(self.config.tensorboard_root_log_dir, timestamp)
        tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=tb_run_log_dir)
        return tensorboard_callback
    
    @property
    def _create_checkpoint_callback(self):
        checkpoint_dir = os.path.dirname(self.config.checkpoint_model_filepath)
        create_directories([checkpoint_dir])
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True,
        )
        return checkpoint_callback
    
    def get_tb_ckpt_callbacks(self):
        tb_callback = self._create_tb_callback
        ckpt_callback = self._create_checkpoint_callback
        return [
            tb_callback, 
            ckpt_callback
        ]

In [10]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path, compile=False)
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"],
        )

    def train_valid_generator(self):
        datagen_kwargs = dict(rescale=1./255, validation_split=0.20)

        dataflow_kwargs = dict(target_size=self.config.params_image_size[:-1], batch_size=self.config.params_batch_size, interpolation="bilinear")

        valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(**datagen_kwargs)

        self.valid_generator = valid_datagen.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                zoom_range=0.2,
                shear_range=0.2,
                **datagen_kwargs
            )
        else:
            train_datagen = valid_datagen

        self.train_generator = train_datagen.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self, callbacks_list: list):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps=self.validation_steps,
            epochs=self.config.params_epochs,
            callbacks=callbacks_list
        )

        self.save_model(path=self.config.trained_model_path, model=self.model)

In [11]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callbacks_config()
    prepare_callbacks = PrepareCallbacksTrainingPipeline(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(
        callbacks_list=callback_list
    )
    
except Exception as e:
    raise CustomException(e, sys)

[2026-05-10 16:30:07,400: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-10 16:30:07,403: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-10 16:30:07,405: INFO: common: created directory at: artifacts]
[2026-05-10 16:30:07,407: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir]
[2026-05-10 16:30:07,408: INFO: common: created directory at: artifacts\prepare_callbacks\tensorboard_log_dir]
[2026-05-10 16:30:07,410: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir]
[2026-05-10 16:30:07,411: INFO: common: created directory at: artifacts\training]
[2026-05-10 16:30:08,500: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
Found 78 images belonging to 2 classes.
Found 312 images belonging to 2 classes.
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 340